In [71]:
YEAR_FILTER = '0113-2'  # 修改為你想篩選的年份
data_path = f'all_predictions_{YEAR_FILTER}.csv'  # 替換為你的數據文件路徑
true_path = f'true_value_{YEAR_FILTER}.csv'

In [72]:
import os
import pandas as pd
import optuna
import numpy as np

INPUT_FOLDER = f'predicted_results_{YEAR_FILTER}/'  # 儲存各系預測結果的資料夾

def merge_csv_files(input_folder, output_file):
    """合併資料夾中的所有 CSV 檔案到一個新的檔案中"""
    all_data = []

    # 讀取資料夾中的所有 CSV 檔案
    for file in os.listdir(input_folder):
        if file.endswith('.csv'):
            file_path = os.path.join(input_folder, file)
            print(f"讀取檔案: {file_path}")
            
            # 讀取 CSV 並加上系所名稱欄位
            df = pd.read_csv(file_path)
            department_name = file.split('_')[0]  # 從檔名提取系所名稱
            df['系所'] = department_name  # 新增一個系所欄位
            all_data.append(df)

    # 合併所有的資料
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)

        # 調整系所欄位到第一欄
        columns = ['系所'] + [col for col in combined_df.columns if col != '系所']
        combined_df = combined_df[columns]

        # 儲存合併後的資料
        combined_df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"已將所有資料合併並儲存至 {output_file}")
    else:
        print("沒有可用的 CSV 檔案進行合併")

if __name__ == '__main__':
    merge_csv_files(INPUT_FOLDER, data_path)

讀取檔案: predicted_results_0113-2/數學系 MATH_predictions.csv
讀取檔案: predicted_results_0113-2/測量系 GM_predictions.csv
讀取檔案: predicted_results_0113-2/統計系 STAT_predictions.csv
讀取檔案: predicted_results_0113-2/資訊系 CSIE_predictions.csv
已將所有資料合併並儲存至 all_predictions_0113-2.csv


In [73]:
INPUT_FOLDER = f'filtered_courses_{YEAR_FILTER}/'  # 儲存這四個檔案的資料夾路徑

def combine_csv_files(input_folder, output_file):
    """結合資料夾中的所有 CSV 檔案為一個檔案，並將系所欄位放到第一欄，並修改欄位名稱"""
    all_data = []

    # 讀取資料夾中的所有 CSV 檔案
    for file in os.listdir(input_folder):
        if file.endswith('.csv'):
            file_path = os.path.join(input_folder, file)
            print(f"讀取檔案: {file_path}")
            
            # 讀取 CSV 並加上系所名稱欄位
            df = pd.read_csv(file_path)
            department_name = file.split('_')[0]  # 從檔名提取系所名稱
            df['系所'] = department_name  # 新增一個系所欄位

            # 將系所欄位移到第一列
            columns = ['系所'] + [col for col in df.columns if col != '系所']
            df = df[columns]

            # 修改欄位名稱
            df.rename(columns={
                '已選人數': '預測選課',
                '剩餘名額': '預測餘額',
                # 根據需要修改更多欄位名稱
            }, inplace=True)

            all_data.append(df)

    # 合併所有的資料
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"已將所有資料合併並儲存至 {output_file}")
    else:
        print("沒有可用的 CSV 檔案進行合併")

if __name__ == '__main__':
    combine_csv_files(INPUT_FOLDER, true_path)

讀取檔案: filtered_courses_0113-2/數學系 MATH_0113-2.csv
讀取檔案: filtered_courses_0113-2/測量系 GM_0113-2.csv
讀取檔案: filtered_courses_0113-2/統計系 STAT_0113-2.csv
讀取檔案: filtered_courses_0113-2/資訊系 CSIE_0113-2.csv
已將所有資料合併並儲存至 true_value_0113-2.csv


In [74]:
def filter_and_merge_true_based_on_pred(true_file, pred_file, output_file):
    """刪除 true.csv 中不在 pred.csv 中的科目名稱和系所名稱組合，並合併 true.csv 中科目名稱相同的資料"""
    # 讀取 true.csv 和 pred.csv
    true_df = pd.read_csv(true_file)
    pred_df = pd.read_csv(pred_file)

    # 確認是否有科目名稱和系所欄位
    required_columns = ['科目名稱', '系所']
    for col in required_columns:
        if col not in true_df.columns or col not in pred_df.columns:
            raise ValueError(f"檔案中缺少必要欄位：{col}")

    # 獲取 pred.csv 中的科目名稱和系所組合
    pred_courses = pred_df[['系所', '科目名稱']].drop_duplicates()
    print(f"預測資料包含 {len(pred_courses)} 個科目名稱和系所組合。")

    # 篩選 true.csv 中的資料
    filtered_true_df = pd.merge(true_df, pred_courses, on=['科目名稱', '系所'])
    print(f"篩選後，真實資料剩餘 {len(filtered_true_df)} 筆資料。")

    filtered_true_df.to_csv(true_path, index=False, encoding='utf-8')

    # 合併 true.csv 中科目名稱相同的資料
    # 假設數字型欄位進行求和，其他欄位取第一個值
    numeric_columns = filtered_true_df[['預測選課','預測餘額']]
    merged_true_df = (
        filtered_true_df.groupby(['系所', '科目名稱'], as_index=False)
        .agg({**{col: 'sum' for col in numeric_columns}, **{col: 'first' for col in filtered_true_df.columns if col not in numeric_columns and col not in ['科目名稱', '系所']}})
    )

    print(f"合併後，真實資料剩餘 {len(merged_true_df)} 筆資料。")

    # 儲存合併後的資料
    merged_true_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"已將合併後的真實資料儲存至 {output_file}")

if __name__ == '__main__':
    filter_and_merge_true_based_on_pred(true_path, data_path, true_path)

預測資料包含 74 個科目名稱和系所組合。
篩選後，真實資料剩餘 79 筆資料。
合併後，真實資料剩餘 74 筆資料。
已將合併後的真實資料儲存至 true_value_0113-2.csv


In [75]:
def calculate_popularity(data_path):
    """直接更新原始檔案中的課程熱門度"""
    # 讀取數據
    df = pd.read_csv(data_path)

    # 確保必要欄位存在
    if '預測選課' not in df.columns or '預測餘額' not in df.columns:
        raise ValueError("數據中缺少 '預測選課' 或 '預測餘額' 欄位。")

    # 計算熱門度（已選課人數比例）
    df['熱門度_比例'] = df['預測選課'] / (df['預測選課'] + df['預測餘額'])

    # 添加熱門度（餘額）
    df['熱門度_餘額'] = df['預測餘額']

    # 更新原始檔案
    df.to_csv(data_path, index=False, encoding='utf-8')
    print(f"熱門度計算完成，結果已更新至 {data_path}")

# 執行更新

calculate_popularity(data_path)
calculate_popularity(true_path)

熱門度計算完成，結果已更新至 all_predictions_0113-2.csv
熱門度計算完成，結果已更新至 true_value_0113-2.csv


In [76]:
def calculate_popularity(df, weight):
    """
    使用權重計算熱門度：
    weight 為定義 1 和定義 2 的加權比例，範圍為 [0, 1]。
    """
    # 定義 1：已選課人數比例
    popularity_ratio = df['預測選課'] / (df['預測選課'] + df['預測餘額'])

    # 定義 2：餘額反向
    popularity_balance = 1 / (1 + df['預測餘額'])

    # 使用加權公式計算總熱門度
    popularity_score = weight * popularity_ratio + (1 - weight) * popularity_balance
    return popularity_score

def objective(trial, df):
    """Optuna 的目標函式"""
    # 生成權重參數
    weight = trial.suggest_float("weight", 0, 1)

    # 計算熱門度
    df['熱門度'] = calculate_popularity(df, weight)

    # 模擬目標排名（假設真實熱門度以 '已選課人數' 排序）
    df['實際排名'] = df['預測選課'].rank(ascending=False)
    df['預測排名'] = df['熱門度'].rank(ascending=False)

    # 計算排名差異（均方誤差）
    rank_diff = np.mean((df['實際排名'] - df['預測排名']) ** 2)

    return rank_diff

def fun1(data_path):
    # 讀取數據
    df = pd.read_csv(data_path)

    # 確保必要欄位存在
    if '預測選課' not in df.columns or '預測餘額' not in df.columns:
        raise ValueError("數據中缺少 '預測選課' 或 '預測餘額' 欄位。")

    # 使用 Optuna 進行優化
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, df), n_trials=50)

    # 獲取最佳參數
    best_weight = study.best_params["weight"]
    print(f"最佳權重：{best_weight}")

    # 使用最佳參數計算熱門度
    df['最佳熱門度'] = calculate_popularity(df, best_weight)

    # 保存結果
    df.to_csv(data_path, index=False, encoding='utf-8')
    print(f"熱門度結果已保存至 {data_path}")

# 執行計算
fun1(data_path)
fun1(true_path)

[I 2025-01-15 16:53:31,597] A new study created in memory with name: no-name-6dfe17c8-169a-4a1b-a1e5-72b7c8c4d27a
[I 2025-01-15 16:53:31,600] Trial 0 finished with value: 537.1486486486486 and parameters: {'weight': 0.6539583865452555}. Best is trial 0 with value: 537.1486486486486.
[I 2025-01-15 16:53:31,600] Trial 1 finished with value: 560.918918918919 and parameters: {'weight': 0.5522682031132361}. Best is trial 0 with value: 537.1486486486486.
[I 2025-01-15 16:53:31,600] Trial 2 finished with value: 503.6216216216216 and parameters: {'weight': 0.7990657994908407}. Best is trial 2 with value: 503.6216216216216.
[I 2025-01-15 16:53:31,609] Trial 3 finished with value: 696.4594594594595 and parameters: {'weight': 0.16491380241601772}. Best is trial 2 with value: 503.6216216216216.
[I 2025-01-15 16:53:31,613] Trial 4 finished with value: 560.918918918919 and parameters: {'weight': 0.5544633066575511}. Best is trial 2 with value: 503.6216216216216.
[I 2025-01-15 16:53:31,617] Trial 5 f

最佳權重：0.9998895005010546
熱門度結果已保存至 all_predictions_0113-2.csv


[I 2025-01-15 16:53:32,005] Trial 43 finished with value: 564.1891891891892 and parameters: {'weight': 0.8851280187124551}. Best is trial 9 with value: 562.9324324324324.
[I 2025-01-15 16:53:32,010] Trial 44 finished with value: 566.8648648648649 and parameters: {'weight': 0.7670906131462104}. Best is trial 9 with value: 562.9324324324324.
[I 2025-01-15 16:53:32,010] Trial 45 finished with value: 587.5 and parameters: {'weight': 0.2658553992611591}. Best is trial 9 with value: 562.9324324324324.
[I 2025-01-15 16:53:32,017] Trial 46 finished with value: 562.9324324324324 and parameters: {'weight': 0.9534766242379092}. Best is trial 9 with value: 562.9324324324324.
[I 2025-01-15 16:53:32,024] Trial 47 finished with value: 567.8243243243244 and parameters: {'weight': 0.7327537478375614}. Best is trial 9 with value: 562.9324324324324.
[I 2025-01-15 16:53:32,028] Trial 48 finished with value: 564.3648648648649 and parameters: {'weight': 0.8238289845644191}. Best is trial 9 with value: 562.9

最佳權重：0.9869442697904742
熱門度結果已保存至 true_value_0113-2.csv


In [77]:
def categorize_popularity_with_percentiles(df, column_name='熱門度'):
    """
    根據熱門度的數值分佈計算分位數，並標記高、中、低。
    :param df: 包含熱門度數據的 DataFrame
    :param column_name: 熱門度欄位名稱
    :return: 帶有熱門度標籤的 DataFrame
    """
    # 計算分位數
    high_threshold = df[column_name].quantile(0.75)
    low_threshold = df[column_name].quantile(0.25)

    print(f"高閾值 (75%): {high_threshold}")
    print(f"低閾值 (25%): {low_threshold}")

    # 分配標籤
    def assign_label(value):
        if value >= high_threshold:
            return '高'
        elif value <= low_threshold:
            return '低'
        else:
            return '中'

    df['熱門度標籤'] = df[column_name].apply(assign_label)
    return df

# 讀取數據
def fun2(data_path):
    df = pd.read_csv(data_path)

    # 假設熱門度已經計算完成，列名為 '熱門度'
    if '熱門度' not in df.columns:
        raise ValueError("數據中缺少 '熱門度' 欄位。")

    # 添加熱門度標籤
    df = categorize_popularity_with_percentiles(df)

    # 保存結果
    df.to_csv(data_path, index=False, encoding='utf-8')
    print(f"熱門度標籤已生成並保存至 {data_path}")

fun2(data_path)
fun2(true_path)

高閾值 (75%): 0.8553051170095382
低閾值 (25%): 0.726347377497896
熱門度標籤已生成並保存至 all_predictions_0113-2.csv
高閾值 (75%): 1.0
低閾值 (25%): 0.6167392383702573
熱門度標籤已生成並保存至 true_value_0113-2.csv


In [78]:
def assign_labels_with_quantiles(df, column_name, high_quantile, low_quantile):
    """
    根據高低分位數給熱門度分配標籤。
    """
    high_threshold = df[column_name].quantile(high_quantile)
    low_threshold = df[column_name].quantile(low_quantile)

    def label(value):
        if value >= high_threshold:
            return '高'
        elif value <= low_threshold:
            return '低'
        else:
            return '中'

    return df[column_name].apply(label)

def evaluate_labeling(df, predicted_label_column, true_label_column):
    """
    計算分類準確性。
    """
    accuracy = (df[predicted_label_column] == df[true_label_column]).mean()
    return 1 - accuracy  # 返回錯誤率，越小越好

def objective(trial, df):
    """
    Optuna 的目標函式。
    """
    # 生成高低分位數參數
    high_quantile = trial.suggest_float("high_quantile", 0.5, 1.0)
    low_quantile = trial.suggest_float("low_quantile", 0.0, 0.5)

    # 確保高閾值大於低閾值
    if high_quantile <= low_quantile:
        return float("inf")

    # 分配標籤
    df["optuna_label"] = assign_labels_with_quantiles(df, "熱門度", high_quantile, low_quantile)

    # 計算分類準確性
    error = evaluate_labeling(df, "optuna_label", "熱門度標籤")
    return error

def fun3(data_path):
    optuna_df = pd.read_csv(data_path)

    # 確保 '熱門度標籤' 存在於數據中
    if '熱門度標籤' not in optuna_df.columns:
        raise ValueError("數據中缺少 '熱門度標籤' 欄位。")

    # 使用 Optuna 進行優化
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, optuna_df), n_trials=50)

    # 獲取最佳分位數參數
    best_params = study.best_params
    print(f"最佳分位數參數：{best_params}")

    # 使用最佳參數重新分配標籤
    optuna_df["最佳標籤"] = assign_labels_with_quantiles(optuna_df, "熱門度", best_params["high_quantile"], best_params["low_quantile"])

    # 保存結果
    optuna_df.to_csv(data_path, index=False, encoding='utf-8')
    print(f"結果已保存至 {data_path}")

# 執行計算
fun3(data_path)
fun3(true_path)

[I 2025-01-15 16:53:32,117] A new study created in memory with name: no-name-aee7ecde-d0c6-46d0-958a-1b649cdcc8e4
[I 2025-01-15 16:53:32,121] Trial 0 finished with value: 0.32432432432432434 and parameters: {'high_quantile': 0.9380474755235582, 'low_quantile': 0.388961554475478}. Best is trial 0 with value: 0.32432432432432434.
[I 2025-01-15 16:53:32,126] Trial 1 finished with value: 0.16216216216216217 and parameters: {'high_quantile': 0.7505337922447879, 'low_quantile': 0.08855033952776936}. Best is trial 1 with value: 0.16216216216216217.
[I 2025-01-15 16:53:32,130] Trial 2 finished with value: 0.2567567567567568 and parameters: {'high_quantile': 0.557503755723681, 'low_quantile': 0.308186216867849}. Best is trial 1 with value: 0.16216216216216217.
[I 2025-01-15 16:53:32,134] Trial 3 finished with value: 0.2567567567567568 and parameters: {'high_quantile': 0.8134578017191549, 'low_quantile': 0.05600837440543144}. Best is trial 1 with value: 0.16216216216216217.
[I 2025-01-15 16:53:3

最佳分位數參數：{'high_quantile': 0.7491216781963691, 'low_quantile': 0.24057697223589733}
結果已保存至 all_predictions_0113-2.csv


[I 2025-01-15 16:53:32,636] Trial 30 finished with value: 0.013513513513513487 and parameters: {'high_quantile': 0.5629159301455756, 'low_quantile': 0.24072047133017627}. Best is trial 19 with value: 0.0.
[I 2025-01-15 16:53:32,649] Trial 31 finished with value: 0.013513513513513487 and parameters: {'high_quantile': 0.6849008701594292, 'low_quantile': 0.270528210409238}. Best is trial 19 with value: 0.0.
[I 2025-01-15 16:53:32,655] Trial 32 finished with value: 0.04054054054054057 and parameters: {'high_quantile': 0.7439373061070694, 'low_quantile': 0.29840910697635026}. Best is trial 19 with value: 0.0.
[I 2025-01-15 16:53:32,655] Trial 33 finished with value: 0.0 and parameters: {'high_quantile': 0.782619460506096, 'low_quantile': 0.25502236541962703}. Best is trial 19 with value: 0.0.
[I 2025-01-15 16:53:32,671] Trial 34 finished with value: 0.013513513513513487 and parameters: {'high_quantile': 0.8870995883840779, 'low_quantile': 0.23364820638466924}. Best is trial 19 with value: 0

最佳分位數參數：{'high_quantile': 0.7787774017870238, 'low_quantile': 0.25266048036863903}
結果已保存至 true_value_0113-2.csv
